<a href="https://colab.research.google.com/github/vanderbarbosa/mestrado/blob/main/03_Pre_Processamento_e_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PASSO 1: INTEGRAÇÃO COM O GOOGLE DRIVE
# ==============================================================================
from google.colab import drive
import os

# Conecta ao Drive (a mesma pasta onde os Robôs 1 e 2 salvaram os dados)
drive.mount('/content/drive')
caminho_base = '/content/drive/MyDrive/Mestrado_PETR4/'

Mounted at /content/drive


In [2]:
# ==============================================================================
# PASSO 2: INSTALAÇÃO E IMPORTAÇÃO DAS BIBLIOTECAS DE DEEP LEARNING
# ==============================================================================
!pip install transformers torch pandas --quiet
print("Bibliotecas de Deep Learning instaladas com sucesso!\n")

import pandas as pd
import torch
from transformers import pipeline

Bibliotecas de Deep Learning instaladas com sucesso!



In [3]:
# ==============================================================================
# PASSO 3: LEITURA DOS DADOS TEXTUAIS BRUTOS
# ==============================================================================
caminho_arquivo_entrada = caminho_base + "base_textual_petr4.csv"

try:
    df_noticias = pd.read_csv(caminho_arquivo_entrada)
    print(f"Sucesso! Base textual carregada. Total de notícias para a IA ler: {len(df_noticias)}\n")
except FileNotFoundError:
    print(f"ERRO: O arquivo não foi encontrado em {caminho_arquivo_entrada}. Verifique se o Robô 2 rodou corretamente.")


Sucesso! Base textual carregada. Total de notícias para a IA ler: 10



In [4]:
# ==============================================================================
# PASSO 4: INSTANCIAÇÃO DO MODELO TRANSFORMER E EXTRAÇÃO DE SENTIMENTO
# ==============================================================================
print("Baixando e alocando o modelo Transformer na GPU. Isso pode levar 1 minuto...")

# Verificando se a GPU está ligada
device = 0 if torch.cuda.is_available() else -1
if device == 0:
    print("GPU NVIDIA detectada e ativada! O processamento será rápido.")
else:
    print("Aviso: GPU não detectada. O processamento usará a CPU e será mais lento.")

# Utilizaremos o XLM-RoBERTa, um Transformer multilíngue pré-treinado no estado da arte
# que compreende português perfeitamente e supera abordagens baseadas em léxicos.
modelo_nlp = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    max_length=512,
    truncation=True,
    device=device
)

def extrair_sentimento(texto):
    try:
        resultado = modelo_nlp(str(texto))
        label = resultado['label']
        score = resultado['score'] # Confiança estatística da IA (0 a 1)

        # Mapeando: LABEL_0 (Negativo), LABEL_1 (Neutro), LABEL_2 (Positivo)
        if label == 'LABEL_0':
            polaridade = -1  # Pessimismo
        elif label == 'LABEL_2':
            polaridade = 1   # Otimismo
        else:
            polaridade = 0   # Neutro

        # Retorna o sentimento ponderado pela confiança da rede neural
        return polaridade * score
    except Exception as e:
        return 0

print("Iniciando a leitura profunda das notícias financeiras...")
# Aplicando o Cérebro Leitor na coluna de Resumos
df_noticias['Sentimento_Bruto'] = df_noticias['Resumo'].apply(extrair_sentimento)


Baixando e alocando o modelo Transformer na GPU. Isso pode levar 1 minuto...
GPU NVIDIA detectada e ativada! O processamento será rápido.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Iniciando a leitura profunda das notícias financeiras...


In [5]:
# ==============================================================================
# PASSO 5: CONSTRUÇÃO DO ÍNDICE DE SENTIMENTO DIÁRIO DA MÍDIA
# ==============================================================================
# A B3 opera em dias úteis, então agrupamos os sentimentos por dia.
df_noticias['Data'] = pd.to_datetime(df_noticias['Data_Coleta']).dt.date
df_indice_diario = df_noticias.groupby('Data')['Sentimento_Bruto'].mean().reset_index()

# Renomeando a coluna para o nosso padrão acadêmico
df_indice_diario.rename(columns={'Sentimento_Bruto': 'Indice_Sentimento_Transformer'}, inplace=True)

print("\nExtração finalizada! Veja as primeiras linhas do nosso Índice Matemático de Sentimento:")
display(df_indice_diario.head())



Extração finalizada! Veja as primeiras linhas do nosso Índice Matemático de Sentimento:


,Data,Indice_Sentimento_Transformer
0,2026-04-07,0.0


In [6]:
# ==============================================================================
# PASSO 6: SALVAMENTO AUTOMATIZADO DO ÍNDICE NO GOOGLE DRIVE
# ==============================================================================
caminho_arquivo_saida = caminho_base + "indice_sentimento_petr4.csv"
df_indice_diario.to_csv(caminho_arquivo_saida, index=False)
print(f"\nVitória! O Índice de Sentimento foi salvo de forma automatizada no seu Drive em: {caminho_arquivo_saida}")


Vitória! O Índice de Sentimento foi salvo de forma automatizada no seu Drive em: /content/drive/MyDrive/Mestrado_PETR4/indice_sentimento_petr4.csv
